## Importación de módulos

In [50]:
import torch
import torch.nn as nn
import math
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from transformers import BertForSequenceClassification, BertTokenizer, DistilBertForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from datasets import load_dataset
from evaluate import load

### Implementación de Clase basada en polinomios de Chebishev

In [42]:
class ChebyshevPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super(ChebyshevPositionalEncoding, self).__init__()
        self.d_model = d_model
        self.max_len = max_len
        
        pe = torch.zeros(max_len, d_model)

        if max_len > 1:
            p = torch.arange(0, max_len, dtype=torch.float32)
            x = -1.0 + 2.0 * p / (max_len - 1)
        else:
            x = torch.zeros(1, dtype=torch.float32)
            
        arccos_x = torch.acos(x) 
        k = torch.arange(0, d_model, dtype=torch.float32)
        
        angles = arccos_x.unsqueeze(1) * k.unsqueeze(0)
        pe = torch.cos(angles)
        
        pe = pe / math.sqrt(d_model)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, input_ids_or_position_ids, *args, **kwargs):
        seq_len = input_ids_or_position_ids.size(1)
        
        return self.pe[:, :seq_len, :]

### Configuración del Dispositivo y Carga del Modelo Preentrenado

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Ejecutando en: {device}")

Ejecutando en: cpu


### Carga de Modelo BERT

In [ ]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

### Aternativa de DistilBert

In [44]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = DistilBertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Sustitución de la Capa de Embeddings Posicionales Original por Chebyshev

In [ ]:
d_model = model.config.hidden_size
max_len = model.config.max_position_embeddings

chebyshev_encoder = ChebyshevPositionalEncoding(d_model=d_model, max_len=max_len)

model.bert.embeddings.position_embeddings = chebyshev_encoder
model.to(device)

### Alternativa usando DistilBERT

In [45]:
d_model = model.config.hidden_size
chebyshev_encoder = ChebyshevPositionalEncoding(d_model=d_model, max_len=512)
model.distilbert.embeddings.position_embeddings = chebyshev_encoder
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): ChebyshevPositionalEncoding()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout)

### Creación de un Dataset de Prueba

In [ ]:
texts = [
    "This movie was absolutely fantastic and brilliant!",
    "Terrible film, completely boring and a waste of time.",
    "Great performance by the lead actor, highly recommended.",
    "Worst plot ever, I hated every minute of it."
]
labels = [1, 0, 1, 0]  # 1: Positivo, 0: Negativo

inputs = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
dataset = TensorDataset(inputs["input_ids"], inputs["attention_mask"], torch.tensor(labels))
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

### Toma de muestra del datase de IMDB

In [46]:
raw_dataset = load_dataset("stanfordnlp/imdb")
train_subset = raw_dataset["train"].shuffle(seed=42).select(range(2000))
test_subset = raw_dataset["test"].shuffle(seed=42).select(range(500))

### Tokenización de los datasets de entrenamiento y de prueba

In [47]:
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_train = train_subset.map(tokenize_fn, batched=True)
tokenized_test = test_subset.map(tokenize_fn, batched=True)

### Formato para uso de tokens en Pytorck

In [48]:
tokenized_train = tokenized_train.remove_columns(["text"]).rename_column("label", "labels")
tokenized_test = tokenized_test.remove_columns(["text"]).rename_column("label", "labels")

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(tokenized_train, batch_size=16, shuffle=True, collate_fn=data_collator)
test_loader = DataLoader(tokenized_test, batch_size=16, collate_fn=data_collator)

### Configuración del optimizador de AdamW

Este algoritmo optimizador se usa normalmente en tranformers, como lo es en este caso BERT, o DistilBERT

In [51]:
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
accuracy_metric = load("accuracy")
f1_metric = load("f1")

### Fine tuning del modelo (BERT con micro dataset)

In [ ]:
model.train()
epochs = 2

print("\nIniciando entrenamiento...")
for epoch in range(epochs):
    total_loss = 0.0
    for batch_input_ids, batch_attention_mask, batch_labels in dataloader:
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_mask = batch_attention_mask.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels
        )
        
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Época {epoch + 1}/{epochs} - Loss Promedio: {total_loss / len(dataloader):.4f}")

### Fine Tuning del modelo (DistilBERT con sub muestreo de IMDB)

In [52]:
epochs = 2
print("--- Inicio del Fine-Tuning ---")

for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    
    for batch in train_loader:
        # Mover lote a CPU/GPU
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        
        # Forward pass (Paso hacia adelante)
        outputs = model(**batch)
        loss = outputs.loss
        
        # Backward pass (Cálculo de gradientes)
        loss.backward()
        
        # Actualización de pesos con AdamW
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)

--- Inicio del Fine-Tuning ---


### Inferencia con una Muestra Nueva

In [ ]:
model.eval()
sample_text = "Worst plot ever, I hated everything about it."
sample_inputs = tokenizer(sample_text, return_tensors="pt").to(device)

with torch.no_grad():
    logits = model(**sample_inputs).logits
    prediction = torch.argmax(logits, dim=-1).item()

clases = {0: "Negativo", 1: "Positivo"}
print(f"\nTexto evaluado: '{sample_text}'")
print(f"Predicción del modelo: {clases[prediction]}")

### Evaluación por época (DistilBERT  + IMDB sub dataset)

In [53]:
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        
        accuracy_metric.add_batch(predictions=predictions, references=batch["labels"])
        f1_metric.add_batch(predictions=predictions, references=batch["labels"])
        
acc = accuracy_metric.compute()["accuracy"]
f1 = f1_metric.compute()["f1"]

print(f"Época {epoch + 1}/{epochs} | Loss Entrenador: {avg_train_loss:.4f} | Val Accuracy: {acc:.4f} | Val F1: {f1:.4f}")

Época 2/2 | Loss Entrenador: 0.4631 | Val Accuracy: 0.7940 | Val F1: 0.7599
